# Level 2 프로젝트 스타터: Autonomous Vision Agent

과제:

> Find and sort the six cubes in the warehouse into their matching destination pads.

Level 2에서는 scene_state, 정확한 entity ID, coordinate go_to를 사용할 수 없습니다. camera observation, set_head, set_velocity, memory, recovery로 이동합니다.

실행 전에 `MENLO_API_KEY`와 `TOKAMAK_API_KEY`를 모두 설정하세요. 프로젝트는 text LLM decision loop를 필수로 요구합니다.


## 사용 방법

아래 코드는 완성된 solution이 아니라 최소 scaffold입니다. SUPPORT CODE는 setup, wrapper, schema, memory 구조를 제공합니다. STUDENT TODO 영역에서 LLM decision, observation, action execution, verification, memory update 전략을 직접 설계하세요.


In [ ]:
# Colab/로컬 setup. 필요하면 한 번 실행하세요.
# %pip install -q "git+https://github.com/asimovinc/hansung-menlo-robotics-workshop.git" python-dotenv opencv-python Pillow matplotlib


## 스타터 Scaffold

이 cell을 실행한 뒤 TODO 영역을 팀 설계에 맞게 수정하세요.


In [ ]:
"""Level 2 프로젝트 스타터입니다.

이 파일은 완성된 해답이 아니라 최소 scaffold입니다.

SUPPORT CODE 영역은 반복해서 작성할 필요가 없는 wrapper, 자료 구조,
schema validation을 제공합니다. STUDENT TODO 영역은 팀이 직접 설계하고,
개선하고, 테스트하고, 발표에서 설명해야 하는 부분입니다.

Level 2 규칙: `scene_state`, 정확한 entity ID, coordinate `go_to`를 사용할 수
없습니다. 카메라 관찰값, `set_head`, `set_velocity`, memory, recovery로
navigation을 구현해야 합니다.
"""

import asyncio
import json
import math
from dataclasses import dataclass, field
from typing import Any

from menlo_runner.llm import ask_vlm
from menlo_runner.perception import detect_color_blobs


# ---------------------------------------------------------------------------
# SUPPORT CODE: 공통 과제 정의와 필수 LLM decision schema
# ---------------------------------------------------------------------------
TASK = "Find and sort the six cubes in the warehouse into their matching destination pads."

DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}
SIGNAGE_NOTE = (
    "A is the conveyor/cube source area, not a destination. "
    "Destination signs are B red, C green, D blue, E yellow."
)

ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


@dataclass
class AgentDecision:
    """LLM이 반환하고 코드가 검증한 고수준 decision입니다."""

    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    """observe-decide-act cycle 사이에 유지하는 agent 상태입니다."""

    delivered_count: int = 0
    delivery_limit: int | None = None
    priority_colors: list[str] = field(default_factory=list)
    held_color: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    search_turns: int = 0
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_colors: list[str] = field(default_factory=list)
    skipped_colors: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    """LLM과 action code에 전달할 compact observation입니다."""

    robot_status: Any
    detections: list[Any]
    note: str = ""
    vlm_summary: str = ""


@dataclass(frozen=True)
class ScannedDetection:
    """head pose를 함께 저장한 색상 detection입니다."""

    color: str
    angle_deg: float
    blob_area: int
    centroid: tuple[int, int]
    bbox: tuple[int, int, int, int]
    head_yaw: float
    head_pitch: float

    @property
    def full_bearing_deg(self) -> float:
        """image angle에 head yaw를 더한 대략적인 body-relative bearing입니다."""
        return self.angle_deg + math.degrees(self.head_yaw)


def parse_agent_decision(text: str) -> AgentDecision | None:
    """LLM의 JSON 응답을 parse하고 필수 schema를 검증합니다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]
    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """robot state를 LLM에 전달하기 좋은 compact text context로 변환합니다."""
    visible = [
        {
            "color": detection.color,
            "angle_deg": detection.angle_deg,
            "full_bearing_deg": round(getattr(detection, "full_bearing_deg", detection.angle_deg), 1),
            "blob_area": detection.blob_area,
            "bbox": detection.bbox,
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "active_color": memory.active_color,
        "stage": memory.stage,
        "delivered_count": memory.delivered_count,
        "delivery_limit": memory.delivery_limit,
        "priority_colors": memory.priority_colors,
        "completed_colors": memory.completed_colors,
        "skipped_colors": memory.skipped_colors,
        "failed_attempts": memory.failed_attempts,
        "last_result": last_result,
        "note": observation.note,
        "signage_note": SIGNAGE_NOTE,
        "vlm_summary": observation.vlm_summary,
    }


# ---------------------------------------------------------------------------
# SUPPORT CODE: Level 2에서 허용되는 SDK wrapper
# ---------------------------------------------------------------------------
# 이 wrapper에는 scene_state, 정답 좌표, 정확한 cube/pad entity ID, go_to를
# 추가하지 마세요.

async def get_robot_status(ctx: Any) -> Any:
    """robot pose, motion status, neck state를 읽습니다."""
    return await ctx.state("robot_status")


async def get_camera_frame(ctx: Any) -> bytes:
    """POV camera frame을 가져옵니다."""
    return await ctx.get_vision("pov")


def build_signage_vlm_prompt(held_color: str | None = None) -> str:
    """고정 창고 표지판을 읽기 위한 VLM prompt입니다."""
    target = ""
    if held_color in DESTINATION_SIGN_RULES:
        target = f" The robot is holding a {held_color} cube, so the target destination sign is {DESTINATION_SIGN_RULES[held_color]}."
    return (
        "Read the floating warehouse signs visible in this robot camera frame. "
        f"{SIGNAGE_NOTE} "
        "Return JSON with visible sign letters, colors, rough left/center/right positions, and confidence."
        + target
    )


async def ask_vlm_about_frame(ctx: Any, prompt: str, *, api_key: str) -> str:
    """현재 POV frame에 대해 project-allowed VLM helper로 질문합니다."""
    jpeg = await get_camera_frame(ctx)
    return ask_vlm(jpeg, prompt, api_key=api_key)


async def perceive(ctx: Any) -> list[Any]:
    """현재 camera frame에서 Workshop 2 color-blob detector를 실행합니다."""
    jpeg = await get_camera_frame(ctx)
    return detect_color_blobs(jpeg)


async def set_head(ctx: Any, *, yaw: float | None = None, pitch: float | None = None) -> Any:
    """walking direction은 바꾸지 않고 camera 방향만 조정합니다."""
    args: dict[str, float] = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    return await ctx.invoke("set_head", args, timeout_s=10)


async def move_velocity(
    ctx: Any,
    *,
    vx: float = 0.0,
    vy: float = 0.0,
    wz: float = 0.0,
    duration_s: float = 1.0,
) -> Any:
    """짧은 body-frame velocity command를 보냅니다."""
    return await ctx.invoke(
        "set_velocity",
        {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
        timeout_s=30,
    )


async def cancel_action(ctx: Any) -> Any:
    """현재 실행 중인 runtime action을 취소합니다."""
    return await ctx.invoke("cancel", {})


async def pick_nearest_cube(ctx: Any) -> Any:
    """의도한 cube 가까이 시각적으로 이동한 뒤 nearest cube를 집습니다."""
    return await ctx.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": "cube"}},
        timeout_s=300,
    )


async def place_nearest_zone(ctx: Any) -> Any:
    """matching pad 가까이 이동한 뒤 nearest zone에 내려놓습니다."""
    return await ctx.invoke("place_entity", {}, timeout_s=300)


def result_summary(result: Any) -> dict[str, Any]:
    """SDK result를 log에 넣기 쉬운 작은 dictionary로 변환합니다."""
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    return {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }


async def scan_head(
    ctx: Any,
    *,
    yaws: tuple[float, ...] = (-0.8, 0.0, 0.8),
    pitch: float = 0.15,
) -> list[Any]:
    """간단한 scan helper입니다. 더 나은 search 전략으로 교체할 수 있습니다."""
    all_detections: list[Any] = []
    for yaw in yaws:
        await set_head(ctx, yaw=yaw, pitch=pitch)
        await asyncio.sleep(0.4)
        for detection in await perceive(ctx):
            all_detections.append(
                ScannedDetection(
                    color=detection.color,
                    angle_deg=detection.angle_deg,
                    blob_area=detection.blob_area,
                    centroid=detection.centroid,
                    bbox=detection.bbox,
                    head_yaw=yaw,
                    head_pitch=pitch,
                )
            )
    return all_detections


# ---------------------------------------------------------------------------
# STUDENT TODO: LLM decision 함수
# ---------------------------------------------------------------------------

async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """LLM으로 다음 고수준 action을 선택하고, 실패 시 stage-aware fallback을 사용합니다."""
    decision_context = build_decision_context(task, observation, memory, last_result)
    system_prompt = "\n".join(
        [
            "You are the high-level planner for a Level 2 vision-only warehouse robot.",
            "Return ONLY one JSON object. Do not include markdown.",
            "Allowed next_action values: " + ", ".join(sorted(ALLOWED_NEXT_ACTIONS)) + ".",
            'Schema: {"next_action": "...", "target_color": null, "reason": "...", "recovery_strategy": null}',
            "Do not output velocity values, coordinates, entity ids, or low-level robot commands.",
            "Use the current stage/memory: need_cube/search_cube -> navigate_to_cube -> pick_cube -> search_pad -> navigate_to_pad -> place_cube -> stop.",
            "If holding a cube, only search/navigate/place the matching destination pad for held_color.",
            "If repeated failures are present, choose recover or skip_target rather than looping forever.",
        ]
    )

    def _safe_stage_decision(parsed: AgentDecision | None) -> AgentDecision | None:
        if parsed is None:
            return None
        if parsed.next_action not in ALLOWED_NEXT_ACTIONS:
            return None
        if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
            return AgentDecision(next_action="stop", reason="Delivery limit reached before LLM action.")
        if memory.held_color:
            if parsed.next_action in {"search_cube", "navigate_to_cube", "pick_cube"}:
                return None
            if parsed.target_color not in {None, memory.held_color}:
                return None
            parsed.target_color = memory.held_color
        else:
            if parsed.next_action in {"search_pad", "navigate_to_pad", "place_cube"}:
                return None
            if parsed.next_action == "pick_cube" and memory.stage != "ready_to_pick":
                return None
        if parsed.next_action == "place_cube" and memory.stage != "ready_to_place":
            return None
        return parsed

    parsed: AgentDecision | None = None
    try:
        from menlo_runner.llm import call_llm  # type: ignore

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(decision_context, ensure_ascii=False)},
        ]
        try:
            reply = await asyncio.to_thread(call_llm, messages)
        except TypeError:
            # 일부 runtime은 api_key keyword를 받는 wrapper를 제공합니다.
            import os

            reply = await asyncio.to_thread(call_llm, messages, api_key=os.environ.get("TOKAMAK_API_KEY", ""))
        parsed = _safe_stage_decision(parse_agent_decision(reply))
    except Exception as exc:
        memory.logs.append({"llm_fallback": type(exc).__name__, "stage": memory.stage})

    if parsed is not None:
        return parsed

    visible = decision_context["visible_targets"]
    repeated_failures = max(memory.failed_attempts.values(), default=0)
    active_failures = sum(v for k, v in memory.failed_attempts.items() if memory.active_color and memory.active_color in k)

    if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
        return AgentDecision(next_action="stop", reason="Delivery limit reached.")
    if repeated_failures >= 4:
        return AgentDecision(next_action="recover", reason="Repeated failures require recovery.", recovery_strategy="wide_backoff_rescan")
    if active_failures >= 6 and not memory.held_color and memory.active_color:
        return AgentDecision(next_action="skip_target", target_color=memory.active_color, reason="Too many failures on this visible target; retarget next cube.")

    if memory.held_color:
        target_color = memory.held_color
        matching_visible = [item for item in visible if item["color"] == target_color]
        best = max(matching_visible, key=lambda item: item["blob_area"], default=None)
        if memory.stage == "ready_to_place":
            return AgentDecision(next_action="place_cube", target_color=target_color, reason="Navigation reported matching pad proximity.")
        if memory.stage in {"approaching_pad", "navigate_pad"} and best:
            return AgentDecision(next_action="navigate_to_pad", target_color=target_color, reason="Matching destination color/sign evidence visible.")
        return AgentDecision(next_action="search_pad", target_color=target_color, reason="Holding cube; search for matching destination pad/sign.")

    if memory.stage == "ready_to_pick" and memory.active_color:
        return AgentDecision(next_action="pick_cube", target_color=memory.active_color, reason="Navigation reported selected cube proximity.")

    # 가장 큰 visible blob을 다음 cube 후보로 사용합니다. 색상 완료 여부로 제외하지 않습니다(동색 큐브 반복 가능).
    best = max(visible, key=lambda item: item["blob_area"], default=None)
    if best:
        return AgentDecision(next_action="navigate_to_cube", target_color=best["color"], reason="Visible cube color selected by camera blob evidence.")
    return AgentDecision(next_action="search_cube", reason="No useful color blob visible; scan for cube.")


# ---------------------------------------------------------------------------
# STUDENT TODO: observation, verification, memory
# ---------------------------------------------------------------------------

async def observe_world(ctx: Any, memory: AgentMemory) -> Observation:
    """현재 stage에 맞춰 camera/head 기반 observation을 수집합니다."""
    robot_status = await get_robot_status(ctx)
    wide_scan_stages = {"need_cube", "search_cube", "holding_cube", "search_pad", "recover"}
    if memory.stage in wide_scan_stages:
        detections = await scan_head(ctx, yaws=(-0.9, -0.45, 0.0, 0.45, 0.9), pitch=0.15)
        note = "wide_scan"
    else:
        await set_head(ctx, yaw=0.0, pitch=0.15)
        await asyncio.sleep(0.2)
        detections = [
            ScannedDetection(
                color=d.color,
                angle_deg=d.angle_deg,
                blob_area=d.blob_area,
                centroid=d.centroid,
                bbox=d.bbox,
                head_yaw=0.0,
                head_pitch=0.15,
            )
            for d in await perceive(ctx)
        ]
        note = "front_frame"
    return Observation(robot_status=robot_status, detections=detections, note=note)


async def verify_outcome(ctx: Any, decision: AgentDecision, action_result: dict[str, Any]) -> dict[str, Any]:
    """SDK 결과와 직후 camera evidence를 함께 묶어 다음 cycle 판단에 제공합니다."""
    result_payload = action_result.get("result") if isinstance(action_result, dict) else None
    if isinstance(result_payload, dict):
        status = result_payload.get("status")
        error = result_payload.get("error")
    elif hasattr(result_payload, "status"):
        status = getattr(result_payload, "status", None)
        error = getattr(getattr(result_payload, "error", None), "message", None)
    else:
        status = action_result.get("status") if isinstance(action_result, dict) else None
        error = action_result.get("error") if isinstance(action_result, dict) else None

    post_status = await get_robot_status(ctx)
    detections: list[Any] = []
    target_seen = False
    max_target_area = 0

    if decision.next_action not in {"stop", "skip_target"}:
        try:
            detections = await scan_head(ctx, yaws=(-0.35, 0.0, 0.35), pitch=0.15)
            target_seen = any(d.color == decision.target_color for d in detections) if decision.target_color else bool(detections)
            max_target_area = max(
                (d.blob_area for d in detections if decision.target_color is None or d.color == decision.target_color),
                default=0,
            )
        except Exception as exc:
            return {
                "ok": False,
                "decision": decision.__dict__,
                "action_result": action_result,
                "error": f"post_action_observation_failed:{type(exc).__name__}",
                "sdk_status": str(status),
            }

    held_color_guess = None
    if decision.next_action == "pick_cube" and detections:
        # After a successful pick the actually held cube often dominates the camera.
        # Use camera evidence rather than the intended target color because nearest-pick can grab another cube.
        likely_held = max(detections, key=lambda d: d.blob_area, default=None)
        if likely_held is not None and likely_held.blob_area >= 12000:
            held_color_guess = likely_held.color

    status_text = str(status or action_result.get("status", "")).lower()
    sdk_ok = (not error) and any(token in status_text for token in ("done", "success", "succeed", "complete", "ok"))

    ok = True
    if decision.next_action in {"navigate_to_cube", "navigate_to_pad"}:
        ok = bool(action_result.get("reached"))
    elif decision.next_action in {"search_cube", "search_pad"}:
        ok = bool(action_result.get("found"))
    elif decision.next_action in {"pick_cube", "place_cube"}:
        ok = sdk_ok
    elif decision.next_action == "recover":
        ok = str(action_result.get("status", "")).startswith("recovered")

    return {
        "ok": ok,
        "decision": decision.__dict__,
        "action_result": action_result,
        "sdk_status": str(status) if status is not None else None,
        "post_visible_count": len(detections),
        "target_seen": target_seen,
        "max_target_area": max_target_area,
        "held_color_guess": held_color_guess,
        "robot_status_type": type(post_status).__name__,
    }


def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """각 cycle 뒤 stage, active/held color, 실패 횟수, 완료 상태를 갱신합니다."""
    ok = bool(verified.get("ok"))
    color = decision.target_color or memory.active_color or memory.held_color
    failure_key = f"{decision.next_action}:{color or 'none'}"

    if ok:
        memory.failed_attempts.pop(failure_key, None)
    elif decision.next_action != "stop":
        memory.failed_attempts[failure_key] = memory.failed_attempts.get(failure_key, 0) + 1

    if decision.next_action == "search_cube":
        # Search only found a visual target; it does not prove near-cube proximity.
        memory.stage = "search_cube"
        memory.search_turns += 1
    elif decision.next_action == "navigate_to_cube":
        memory.active_color = decision.target_color or memory.active_color
        memory.stage = "ready_to_pick" if ok else "search_cube"
    elif decision.next_action == "pick_cube":
        if ok:
            picked_color = verified.get("held_color_guess") or color
            memory.held_color = picked_color
            memory.active_color = picked_color
            memory.stage = "holding_cube"
            memory.search_turns = 0
            if picked_color != color:
                memory.logs.append({"held_color_corrected_from_camera": {"intended": color, "actual": picked_color}})
        else:
            memory.stage = "search_cube"
    elif decision.next_action == "search_pad":
        memory.stage = "approaching_pad" if ok else "search_pad"
        memory.search_turns += 1
    elif decision.next_action == "navigate_to_pad":
        memory.stage = "ready_to_place" if ok else "search_pad"
    elif decision.next_action == "place_cube":
        if ok:
            delivered_color = memory.held_color or color
            memory.delivered_count += 1
            if delivered_color and delivered_color not in memory.completed_colors:
                memory.completed_colors.append(delivered_color)
            memory.held_color = None
            memory.active_color = None
            memory.search_turns = 0
            memory.stage = "done" if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit else "need_cube"
        else:
            memory.stage = "search_pad"
    elif decision.next_action == "recover":
        memory.stage = "search_pad" if memory.held_color else "search_cube"
    elif decision.next_action == "skip_target":
        if color and color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        memory.active_color = None
        memory.stage = "holding_cube" if memory.held_color else "need_cube"
    elif decision.next_action == "stop":
        memory.stage = "done"

    memory.logs.append(
        {
            "stage": memory.stage,
            "active_color": memory.active_color,
            "held_color": memory.held_color,
            "delivered_count": memory.delivered_count,
            "observation": {
                "visible_count": len(observation.detections),
                "note": observation.note,
                "colors": sorted({getattr(d, "color", "unknown") for d in observation.detections}),
            },
            "llm_decision": decision.__dict__,
            "verified": verified,
            "failed_attempts": dict(memory.failed_attempts),
        }
    )


# ---------------------------------------------------------------------------
# LEVEL 2 STUDENT TODO: vision-only action 구현
# ---------------------------------------------------------------------------
# Level 2에서는 go_to를 호출하면 안 됩니다. camera observation, set_head,
# set_velocity, memory, recovery behavior로 navigation을 구현하세요.

async def visual_search(ctx: Any, target_color: str | None = None) -> bool:
    """head scan과 짧은 body rotation으로 target color blob을 찾습니다."""
    scan_patterns = [
        (-0.95, -0.55, -0.2, 0.2, 0.55, 0.95),
        (-0.7, -0.25, 0.25, 0.7),
        (-0.9, 0.0, 0.9),
    ]
    for idx, yaws in enumerate(scan_patterns):
        detections = await scan_head(ctx, yaws=yaws, pitch=0.15)
        candidates = [d for d in detections if target_color is None or d.color == target_color]
        if candidates:
            best = max(candidates, key=lambda d: d.blob_area)
            yaw = max(-0.8, min(0.8, math.radians(getattr(best, "full_bearing_deg", best.angle_deg))))
            await set_head(ctx, yaw=yaw, pitch=0.15)
            return True
        turn = 0.38 if idx % 2 == 0 else -0.38
        await move_velocity(ctx, vx=0.10, wz=turn, duration_s=0.9)
    return False


def _pad_candidate_score(detection: Any) -> tuple[int, int]:
    """들고 있는 큐브로 보이는 과대/하단 blob보다 멀리 보이는 pad/sign 후보를 우선합니다."""
    x, y, w, h = getattr(detection, "bbox", (0, 0, 0, 0))
    area = getattr(detection, "blob_area", 0)
    cx, cy = getattr(detection, "centroid", (0, 0))
    # Held cube는 보통 화면 하단/중앙에 매우 크게 잡힙니다. pad/sign 탐색에서는 감점합니다.
    held_like_penalty = 1 if ((area > 45000 and cy > 180) or cy > 560) else 0
    # 목적지 sign/pad 후보는 너무 큰 held-cube blob보다 작거나 중간 크기인 blob을 먼저 봅니다.
    usable_area = min(area, 45000)
    return (-held_like_penalty, usable_area)


async def visual_navigate_to_pad(ctx: Any, target_color: str | None) -> bool:
    """held cube blob을 목적지로 착각하지 않도록 pad/sign 후보를 찾아 접근합니다."""
    if target_color is None:
        return False

    await set_head(ctx, yaw=0.0, pitch=0.05)
    await asyncio.sleep(0.2)
    centered_tolerance_deg = 12.0
    seen_good_candidate = 0

    for step in range(22):
        # pad/sign은 바닥 큐브보다 멀거나 위쪽에 보일 수 있어 좁은 head scan을 유지합니다.
        detections = await scan_head(ctx, yaws=(-0.45, 0.0, 0.45), pitch=0.05)
        same_color = [d for d in detections if d.color == target_color]
        # 너무 큰 하단 blob(들고 있는 큐브 가능성)은 제외하되, 후보가 없으면 전체에서 선택합니다.
        filtered = []
        for d in same_color:
            cx, cy = getattr(d, "centroid", (0, 0))
            area = getattr(d, "blob_area", 0)
            # cy가 매우 아래쪽이면 들고 있는 큐브/발밑 blob일 가능성이 높아 pad 후보에서 제외합니다.
            if not ((area > 45000 and cy > 180) or cy > 560):
                filtered.append(d)
        target = max(filtered or same_color, key=_pad_candidate_score, default=None)

        if target is None:
            print(f"  pad-nav: no {target_color} pad/sign candidate, sweeping")
            await move_velocity(ctx, vx=0.12, wz=0.35, duration_s=0.9)
            continue

        bearing = getattr(target, "full_bearing_deg", target.angle_deg)
        area = target.blob_area
        cx, cy = target.centroid
        held_like = (area > 45000 and cy > 180) or cy > 560
        print(f"  pad-nav: {target_color} bearing={bearing:+.1f} area={area} cy={cy} held_like={held_like}")

        if held_like:
            # 정면 하단의 held cube/발밑 blob만 보이면 목적지가 아니므로 몸을 틀어 sign/pad를 찾습니다.
            await move_velocity(ctx, vx=0.10, wz=0.35, duration_s=0.8)
            continue

        if abs(bearing) > centered_tolerance_deg:
            wz = -0.35 if bearing > 0 else 0.35
            await move_velocity(ctx, vx=0.14, wz=wz, duration_s=0.75)
        else:
            seen_good_candidate += 1
            if area >= 18000 and seen_good_candidate >= 2:
                await move_velocity(ctx, vx=0.12, wz=0.0, duration_s=0.5)
                return True
            await move_velocity(ctx, vx=0.28, wz=0.0, duration_s=0.9)

    return False


async def visual_navigate_to_target(ctx: Any, target_color: str | None) -> bool:
    """정면 color blob angle/area feedback으로 실제 전진이 보이게 접근합니다."""
    if target_color is None:
        return False

    arrival_area = 7000
    centered_tolerance_deg = 10.0
    last_seen_area = 0

    # Search 단계에서 이미 target을 찾았으므로, 접근 단계에서는 머리 스캔보다
    # 정면 카메라 feedback을 빠르게 반복해야 실제 이동이 누적됩니다.
    await set_head(ctx, yaw=0.0, pitch=0.15)
    await asyncio.sleep(0.2)

    for step in range(16):
        raw = await perceive(ctx)
        detections = [
            ScannedDetection(
                color=d.color,
                angle_deg=d.angle_deg,
                blob_area=d.blob_area,
                centroid=d.centroid,
                bbox=d.bbox,
                head_yaw=0.0,
                head_pitch=0.15,
            )
            for d in raw
            if d.color == target_color
        ]
        target = max(detections, key=lambda d: d.blob_area, default=None)

        if target is None:
            print(f"  navigate: lost {target_color}, search turn")
            await move_velocity(ctx, vx=0.18, wz=0.30, duration_s=0.8)
            continue

        angle = target.angle_deg
        area = target.blob_area
        last_seen_area = max(last_seen_area, area)
        print(f"  navigate: {target_color} angle={angle:+.1f} area={area}")

        if area >= arrival_area and abs(angle) <= 18.0:
            await move_velocity(ctx, vx=0.10, wz=0.0, duration_s=0.35)
            return True

        if abs(angle) > centered_tolerance_deg:
            wz = -0.35 if angle > 0 else 0.35
            await move_velocity(ctx, vx=0.18, wz=wz, duration_s=0.8)
        else:
            speed = 0.45 if area < arrival_area * 0.55 else 0.25
            await move_velocity(ctx, vx=speed, wz=0.0, duration_s=0.9)

    return last_seen_area >= arrival_area * 0.80


async def recover_motion(ctx: Any, memory: AgentMemory, reason: str | None = None) -> dict[str, Any]:
    """반복 실패 횟수에 따라 후진, 회전, 재스캔 폭을 조절합니다."""
    worst_failure = max(memory.failed_attempts.values(), default=0)
    await cancel_action(ctx)
    await move_velocity(ctx, vx=-0.18, duration_s=0.8)
    turn = 0.35 if worst_failure % 2 == 0 else -0.35
    await move_velocity(ctx, vx=0.10, wz=turn, duration_s=1.1)
    detections = await scan_head(ctx, yaws=(-0.95, -0.35, 0.35, 0.95), pitch=0.15)
    target_color = memory.held_color or memory.active_color
    target_seen = any(d.color == target_color for d in detections) if target_color else bool(detections)
    status = "recovered" if target_seen else "recovered_no_target_seen"
    return {
        "action": "recover",
        "reason": reason,
        "status": status,
        "visible_count": len(detections),
        "target_seen": target_seen,
        "worst_failure": worst_failure,
    }


async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """검증된 LLM decision 하나를 Level 2 robot action으로 변환합니다.

    TODO:
    - go_to 없이 search/navigation을 구현하세요.
    - 의도한 cube 가까이 시각적으로 이동한 뒤 pick하세요.
    - matching pad 가까이 시각적으로 이동한 뒤 place하세요.
    - target을 잃거나 이동에 실패하면 recovery를 사용하세요.
    """
    if decision.next_action in {"search_cube", "search_pad"}:
        found = await visual_search(ctx, decision.target_color)
        return {"action": decision.next_action, "found": found}

    if decision.next_action == "navigate_to_cube":
        reached = await visual_navigate_to_target(ctx, decision.target_color)
        return {"action": decision.next_action, "reached": reached}

    if decision.next_action == "navigate_to_pad":
        reached = await visual_navigate_to_pad(ctx, decision.target_color)
        return {"action": decision.next_action, "reached": reached}

    if decision.next_action == "pick_cube":
        if memory.stage != "ready_to_pick" or not memory.active_color:
            return {"action": "pick_cube", "status": "blocked_precondition", "reason": "not visually confirmed near selected cube"}
        result = await pick_nearest_cube(ctx)
        return {"action": "pick_cube", "result": result_summary(result)}

    if decision.next_action == "place_cube":
        if memory.stage != "ready_to_place" or not memory.held_color:
            return {"action": "place_cube", "status": "blocked_precondition", "reason": "not visually confirmed near matching pad"}
        result = await place_nearest_zone(ctx)
        return {"action": "place_cube", "result": result_summary(result)}

    if decision.next_action == "recover":
        return await recover_motion(ctx, memory, decision.recovery_strategy)

    if decision.next_action == "skip_target":
        return {"action": "skip_target", "status": "skipped"}

    return {"action": decision.next_action, "status": "no_op"}


async def run_agent(ctx: Any, *, max_cycles: int = 80) -> AgentMemory:
    """observe-LLM-act-verify loop with delivery/retry stop conditions."""
    memory = AgentMemory(delivery_limit=6)
    last_result: dict[str, Any] | None = None

    for cycle in range(1, max_cycles + 1):
        print(f"\n[Level 2] Cycle {cycle}")
        if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
            memory.stage = "done"
            break
        if max(memory.failed_attempts.values(), default=0) >= 7:
            recovered = await recover_motion(ctx, memory, "max_failure_guard")
            memory.logs.append({"guard_recovery": recovered, "failed_attempts": dict(memory.failed_attempts)})
            if max(memory.failed_attempts.values(), default=0) >= 9:
                memory.logs.append({"stop_reason": "too_many_repeated_failures", "failed_attempts": dict(memory.failed_attempts)})
                break

        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(TASK, observation, memory, last_result)
        print("LLM decision:", decision)

        if decision.next_action == "stop":
            update_memory(memory, observation, decision, {"ok": True, "stop_reason": decision.reason})
            break

        action_result = await execute_decision(ctx, decision, observation, memory)
        print("Action result:", action_result)
        verified = await verify_outcome(ctx, decision, action_result)
        print("Verified:", {"ok": verified.get("ok"), "target_seen": verified.get("target_seen"), "max_target_area": verified.get("max_target_area"), "sdk_status": verified.get("sdk_status"), "held_color_guess": verified.get("held_color_guess")})
        update_memory(memory, observation, decision, verified)
        last_result = verified

    return memory

async def run(ctx: Any) -> None:
    print(TASK)
    print("Running Level 2 autonomous-vision project starter")
    memory = await run_agent(ctx)
    print("\nRun complete.")
    print(f"Delivered count: {memory.delivered_count}")
    print("Logs:")
    for item in memory.logs:
        print(item)


## 로봇 연결

Prompt가 나오면 viewer URL을 Chrome에서 여세요.


In [ ]:
from menlo_runner.config import load_config
from menlo_runner.context import open_robot_context

config = load_config(require_tokamak=True)
ctx = await open_robot_context(config, name_prefix="level-2-notebook-ko")
print(ctx.viewer_url)


## Agent 실행

현재 실험에 필요한 TODO를 충분히 구현한 뒤 실행하세요.


In [ ]:
memory = await run_agent(ctx)
memory.logs


In [ ]:
# 종료할 때 cleanup하세요.
# await ctx.close()
